In [23]:
import numpy as np
from scipy.stats import pearsonr
from scipy.sparse import load_npz
from sklearn.metrics.pairwise import cosine_similarity

In [24]:
# Carguemos la matriz usuarios-ítems: 
ui_matrix = load_npz('../data/processed/ui_matrix_csr.npz').astype('uint16')

In [25]:
# Para mejorar la eficiencia, crearemos un índice de la matriz para el cálculo posterior de KNN: 

def knn_index(ui_matrix, apply_log=True):
    """
    Preprocesa una vez la matriz para consultas kNN rápidas.
    """
    X = ui_matrix.tocsr().astype(np.float32)

    # Aplicamos escala logarítmica para reducir el efecto de playcounts extremos: 
    if apply_log:
        X = X.copy()
        X.data = np.log1p(X.data)

    # Obtenemos normas L2 por usuario para coseno: 
    norms = np.sqrt(X.multiply(X).sum(axis=1)).A1.astype(np.float32)
    norms[norms == 0] = 1.0

    # Construimos una matriz binaria (donde valores distintos de 0 valen 1) 
    # para facilitar posteriormente el cálculo del número de canciones en común
    # entre dos usuarios al multiplicar B por su traspuesta:
    B = X.copy()
    B.data = np.ones_like(B.data, dtype=np.float32)

    # Calculamos la media por usuario (solo sobre ítems escuchados):
    row_sums = np.asarray(X.sum(axis=1)).ravel().astype(np.float32)
    row_nnz = np.diff(X.indptr)
    user_means = np.divide(
        row_sums,
        row_nnz,
        out=np.zeros_like(row_sums, dtype=np.float32),
        where=row_nnz != 0
    )

    # Devolvemos en forma de diccionario el índice resultante: 
    return {"X": X, "B": B, "norms": norms, "user_means": user_means}

index = knn_index(ui_matrix)
index

{'X': <Compressed Sparse Row sparse matrix of dtype 'float32'
 	with 9711301 stored elements and shape (962037, 30459)>,
 'B': <Compressed Sparse Row sparse matrix of dtype 'float32'
 	with 9711301 stored elements and shape (962037, 30459)>,
 'norms': array([0.98025817, 1.2005662 , 2.0682583 , ..., 0.6931472 , 3.4905324 ,
        2.6569564 ], shape=(962037,), dtype=float32),
 'user_means': array([0.6931472, 0.6931472, 1.1337324, ..., 0.6931472, 0.9552511,
        1.6290482], shape=(962037,), dtype=float32)}

In [26]:
# Creamos ahora la función que calcula los k vecinos mediante el uso del índice creado y 
# optimizado con operaciones de álgebra lineal y vectorizaciones:

def knn_users(index, target_user, k=10, min_common=3, shrink_beta=10.0, positive_only=True):
    """
    kNN de usuarios con coseno vectorizado en sparse.
    Devuelve lista [(user_id, sim), ...].
    """
    X = index["X"]
    B = index["B"]
    norms = index["norms"]

    # Calculamos X[user] * X^t para obtener el producto escalar de ese usuario con los demás. 
    # Dividimos después por la multiplicación de sus  respectivas normas, obteniendo así la 
    # similitud coseno de ese usuario con el resto:
    dots = (X[target_user] @ X.T).toarray().ravel()
    sims = dots / (norms[target_user] * norms + 1e-8)

    # Mediante el uso de la matriz binaria, obtenemos el número de ítems en común del
    # usuario objetivo con respecto a cada uno de los otros usuarios:
    common = (B[target_user] @ B.T).toarray().ravel()

    # Aplicamos Shrinkage, que penaliza similaridades con pocos ítems en común: 
    if shrink_beta is not None and shrink_beta > 0:
        sims *= (common / (common + shrink_beta))

    # Filtramos por aquellos que no cumplen con min_common:
    sims[common < min_common] = -np.inf
    sims[target_user] = -np.inf
    if positive_only:
        sims[sims <= 0] = -np.inf

    valid = np.isfinite(sims)
    if not valid.any():
        return []
    k_eff = min(k, int(valid.sum()))

    # Escogemos los k valores más elevados usando argpartition (muy eficiente pues 
    # no ordena todo el array, solo obtiene los k mayores valores):
    top_idx = np.argpartition(sims, -k_eff)[-k_eff:]
    top_idx = top_idx[np.argsort(sims[top_idx])[::-1]]

    return [(int(u), float(sims[u])) for u in top_idx]

# Ejemplo de prueba: 
target_user = 100
neighbours = knn_users(index, target_user, min_common=1)
neighbours

[(640015, 0.25667113065719604),
 (56451, 0.2561090588569641),
 (9260, 0.23327118158340454),
 (354554, 0.21642901003360748),
 (806205, 0.2123730331659317),
 (841025, 0.20396186411380768),
 (527627, 0.19985203444957733),
 (652960, 0.19985201954841614),
 (65327, 0.19985201954841614),
 (953511, 0.19985201954841614)]

In [28]:
# Creamos una versión vectorizada del deviation from mean creado anteriormente: 

def deviation_from_mean_prediction(ui_matrix, user, item, neighbours, user_means):
    """
    Versión vectorizada de la predicción.
    """
    base = float(user_means[user])
    if not neighbours:
        return base

    neigh_ids = np.fromiter((u for u, _ in neighbours), dtype=np.int32)
    sims = np.fromiter((s for _, s in neighbours), dtype=np.float32)

    # Ratings del item para todos los vecinos de una vez:
    r_vi = ui_matrix[neigh_ids, item].toarray().ravel()
    mask = r_vi > 0
    if not np.any(mask):
        return base

    neigh_ids = neigh_ids[mask]
    sims = sims[mask]
    r_vi = r_vi[mask]

    num = float(np.dot(sims, r_vi - user_means[neigh_ids]))
    den = float(np.abs(sims).sum())
    if den == 0:
        return base

    pred = base + (num / den)
    return float(max(0.0, pred))

# Ejemplo de prueba: 
user = 100
item = 33
user_means = index['user_means']
deviation_from_mean_prediction(ui_matrix, user, item, neighbours, user_means)

0.6931471824645996